## Código dado

In [29]:
# !pip install tensorflow matplotlib
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__ if hasattr(keras, "__version__") else "tf.keras")

TensorFlow: 2.21.0
Keras: 3.14.1


## Dataset CIFAR-10

In [30]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
class_names = np.array([
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
])


x_train = x_train[:1000].astype("float32")/255.0
y_train = y_train[:1000].astype("float32")/255.0

x_test = x_test[:1000].astype("float32")/255.0
y_test = y_test[:1000].astype("float32")/255.0


print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

(1000, 32, 32, 3) (1000, 1)
(1000, 32, 32, 3) (1000, 1)


In [31]:
def make_class(images, labels, batch_size = 64, training=True):
  ds = tf.data.Dataset.from_tensor_slices((images, labels))
  if training :
    ds = ds.shuffle(4*batch_size)

  return ds

train_ds = make_class(x_train, y_train, training=True)
test_ds = make_class(x_test, y_test, training=False)



---->
entrada  ||| [bloque1] [bloque2] [bloque3] [bloque4]  |||  [bloque1] [bloque2] [bloque3] [bloque4]
       _________________________| __________________|~~~~~~~~~~~~~~~~~~~~~~~~~|




# **Arquitectura resnet a mano**


## Definimos los bloques

In [32]:
def conv_relu(x, filters, kernel_size=3, stride=1):
    x = layers.Conv2D(
        filters=filters,
        kernel_size=kernel_size,
        strides=stride,
        padding="same",
        kernel_initializer="he_normal"
    )(x)

    x=layers.BatchNormalization()(x)
    x=layers.Activation("relu")(x)

    return x

def resid_block(x, filters, stride, projection):
    x_atajo = x
    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        strides=stride,
        padding="same",
        kernel_initializer="he_normal"
    )(x)
    x=layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)


    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
    )(x)
    x=layers.BatchNormalization()(x)

    if projection:
        x_atajo = layers.Conv2D(
          filters=filters,
          kernel_size=1,
          strides=stride,
          padding="same",
          kernel_initializer="he_normal"
        )(x_atajo)
        x_atajo = layers.BatchNormalization()(x_atajo)


    x=layers.Add()([x, x_atajo])
    x=layers.Activation("relu")(x)

    return x





## ResNet pequeña

3 bloques con 16f -> 3 bloques 32f/2-> 3 bloques 64f/2 -> GlobalAveragePooling2D -> Dense(10)

In [33]:
def build_resnet(input_shape, num_classes, filter, blocks):
  input = keras.Input(shape=input_shape)
  x = conv_relu(input, filters=filter[0], kernel_size=3, stride=1)


  for indice_stage, (filters, n_blocks) in enumerate(zip(filter, blocks), start=0):
    for indice_bloque in range(n_blocks):
      is_first_block = indice_bloque==0
      is_first_block_network = indice_stage==0



      stride = 2 if indice_stage > 0 and indice_bloque == 0 else 1
      use_projection = stride != 1 or x.shape[-1] != filters


      x = resid_block(x, filters, stride, use_projection)


  x = layers.GlobalAveragePooling2D()(x)
  outputs = layers.Dense(num_classes, activation="softmax")(x)

  return keras.Model(inputs=input, outputs=outputs, name="resnet_cifar_manual")


nuestra_red_resnet = build_resnet((32,32,3), 10, (16,32,64), (3,3,3))
nuestra_red_resnet.summary()


Model: "resnet_cifar_manual"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_12      │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_42 (Conv2D)  │ (None, 32, 32,    │        448 │ input_layer_12[0… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_42[0][0]   │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_38       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_43 (Conv2D)  │ (None, 32, 32,    │      2,320 │ activation_38[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_43[0][0]   │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_39       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_44 (Conv2D)  │ (None, 32, 32,    │      2,304 │ activation_39[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_44[0][0]   │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_18 (Add)        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 16)               │            │ activation_38[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_40       │ (None, 32, 32,    │          0 │ add_18[0][0]      │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_45 (Conv2D)  │ (None, 32, 32,    │      2,320 │ activation_40[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_45[0][0]   │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_41       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_46 (Conv2D)  │ (None, 32, 32,    │      2,304 │ activation_41[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_46[0][0]   │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_19 (Add)        │ (None, 32, 32,    │          0 │ batch_normalizat

 Total params: 274,490 (1.05 MB)

 Trainable params: 272,922 (1.04 MB)

 Non-trainable params: 1,568 (6.12 KB)

In [35]:
nuestra_red_resnet.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

train_ds = train_ds.batch(64)
test_ds = test_ds.batch(3)
history_resnet = nuestra_red_resnet.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5
)


Epoch 1/5


ValueError: Input 0 with name 'input_layer_12' of layer 'resnet_cifar_manual' is incompatible with the layer: expected shape=(None, 32, 32, 3), found shape=(None, None, 32, 32, 3)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history_resnet.history['accuracy'], label='train_acc')
plt.plot(history_resnet.history['val_accuracy'], label='val_acc')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## Resnet keras

In [ ]:
from tensorflow.keras.applications import ResNet50

base_model = ResNet50(
    include_top=False,
    weights=None,
    input_shape=(32, 32, 3),
    pooling="avg"
)

outputs = layers.Dense(
    10,
    activation="softmax"
    )(base_model.output)

model = keras.Model(
    inputs=base_model.input,
    outputs=outputs,
    name="resnet_cifar_builtin"
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_resnet = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5
)


# Transfer learning con ResNet50

## Preparar datos para ResNet50

In [ ]:
def preprocess_resnet50(image, label):
    image = tf.image.resize(image, (96, 96))
    image =image*255.0
    image = keras.applications.resnet50.preprocess_input(image)
    return image, label

In [ ]:
transfer_train_df = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(4*64, seed=456)
    .map(preprocess_resnet50)
    .batch(64)
)

transfer_test_df = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(preprocess_resnet50)
    .batch(64)
)

## Transfer learning

Feature Extraction

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

base_model = keras.applications.ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(96, 96, 3),
    pooling="avg"
)

base_model.trainable = False

inputs = keras.Input(shape=(96, 96, 3))
x = base_model(inputs, training=False)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(10, activation="softmax")(x)

transfer_model = keras.Model(inputs=inputs, outputs=outputs)
transfer_model.summary()

In [ ]:
transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)   

history_resnet_transfer_learning = transfer_model.fit(
    transfer_train_df,
    epochs=3,
    validation_data=transfer_test_df
)

## Fine tuning

## Comparación final